# NYC Yellow Taxi Fare Prediction: Decision Tree, Bagging and Random Forest (Zhanel)

This notebook runs the tree ensemble models on the same trips and the same chronological train / validation / test split as the boosting and linear models.

**What it trains** (each one twice: *Pre-trip* and *Post-trip*)
- Single decision tree (for comparison)
- **Bagging**: many decision trees, each trained on a bootstrap sample of trips; predictions are averaged
- **Random forest**: bagging plus a random subset of features at each split, so the trees are less alike
- The same median baseline as the other models

**Pre-trip** uses pickup zone, drop-off zone and pickup time (what a rider knows in advance).
**Post-trip** adds actual distance and duration (known only after the ride).

All the real code lives in `random_forest.py` in the repo root (it reuses the loading, cleaning and split functions from `linear_models.py`). This notebook just calls it, so the notebook and the script always give the same answers.

## Step 1: Setup
On Google Colab this downloads the repo. On your own computer it finds the repo root automatically.

In [1]:
import os, sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("MIS545-NYCTaxi").exists():
        subprocess.run(["git", "clone", "https://github.com/nataperezzr/MIS545-NYCTaxi.git"], check=True)
    os.chdir("MIS545-NYCTaxi")
else:
    p = Path.cwd()
    while p != p.parent and not (p / "random_forest.py").exists():
        p = p.parent
    os.chdir(p)
sys.path.insert(0, os.getcwd())
print("Working in:", os.getcwd())

import warnings
import pandas as pd
import linear_models as lm
import random_forest as rfm
warnings.filterwarnings("ignore", category=UserWarning)

Working in: /content/MIS545-NYCTaxi


## Step 2: Settings
Change these, then run the cells below.

In [2]:
DATA_PATH = "data/yellow_tripdata_2026-05.parquet"
SAMPLE_ROWS = 0           # 0 = whole month (default, roughly 20-40 minutes). Use 100_000 for a quick code check
N_TREES = 100             # trees in the final bagging and random forest models
FARE_CUTOFF = 35.0        # dollars, for the "$35 or more?" columns (same as the linear models)
CARD_ONLY = False         # keep False to match the other models' trips
USE_SYNTHETIC = False     # True = fake demo data, only to test that the notebook runs

rfm.N_TREES = N_TREES

## Step 3: Load the data
If the parquet file is not in `data/`, the next cell downloads it from the NYC TLC (about 70 MB). On Colab, if the download is blocked, it asks you to upload the file.

In [3]:
if USE_SYNTHETIC:
    print("Using SYNTHETIC demo data. Results are not real.")
    raw = lm.make_synthetic()
else:
    if not Path(DATA_PATH).exists():
        try:
            lm.download_data(DATA_PATH)
        except Exception as err:
            if "google.colab" not in sys.modules:
                raise SystemExit(f"Could not download the file ({err}). Put yellow_tripdata_2026-05.parquet in data/ and re-run.")
            from google.colab import files
            uploaded = files.upload()
            Path(DATA_PATH).parent.mkdir(exist_ok=True)
            os.replace(next(iter(uploaded)), DATA_PATH)
    raw = lm.load_data(DATA_PATH, SAMPLE_ROWS)
display(raw.head())

Total rows in file: 4,090,836


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount
0,2026-05-01 00:04:59,2026-05-01 00:32:48,1.0,7.51,138,37,2,35.9
1,2026-05-01 00:37:05,2026-05-01 00:59:47,1.0,6.14,138,237,1,27.5
2,2026-05-01 00:34:05,2026-05-01 00:54:02,1.0,2.40,249,232,1,19.1
3,2026-05-01 00:55:07,2026-05-01 01:02:43,0.0,1.20,232,114,1,9.3
4,2026-05-01 00:44:13,2026-05-01 00:44:13,2.0,0.86,140,237,1,7.2


## Step 4: Clean the data
Same rules as the other models (positive fare/distance, 1 to 240 minutes, valid zones, May 2026 pickups only) so the rows match.

In [4]:
clean = lm.clean_data(raw, card_only=CARD_ONLY)

Trips with pickup outside May 2026 (wrong dates): 14 (removed)
Rows before cleaning: 4,090,836 | after: 3,899,097 | excluded: 191,739


## Step 5: Train, tune and evaluate
This one cell does everything: builds the features, splits the trips by time (70% / 15% / 15%), picks the random forest settings (`min_samples_leaf`, `max_features`) using the **validation** period only, then scores every model once on the **test** period. It also runs 5-fold cross-validation and draws the predicted-vs-actual plots, feature importance plots and learning curves. Plots are also saved in `results/plots/`.

In [ ]:
results, tuning, importances, models = rfm.run_experiment(
    clean, cutoff=FARE_CUTOFF, out_dir="results", show_plots=True)

Train: 2,729,367 | Validation: 584,865 | Test: 584,865
Train dates: 2026-05-01 00:00:00 to 2026-05-21 20:15:15
Test dates:  2026-05-27 15:26:30 to 2026-05-31 23:59:59


## Step 6: Results
### Dollar-fare models (lower MAE / RMSE is better, higher R² is better)
- **Train MAE vs. MAE (test):** a much lower training error than test error means the model is overfitting.
- The last four metric columns turn each predicted fare into a "$35 or more?" answer, so these models can be compared with the logistic model on the same scale.

In [ ]:
display(results.round(3))

### Random forest tuning (validation period)

In [ ]:
for approach, table in tuning.items():
    print(approach)
    display(table.round(3))

### Which inputs matter most?
- **Impurity importance:** how much each input reduces error inside the trees (built in, fast).
- **Permutation importance:** how much the test MAE gets worse, in dollars, when that input is shuffled (more reliable).

In [ ]:
for approach, table in importances.items():
    print(approach)
    display(table.round(3))

### Learning curve numbers (random forest)
If training error and validation error are both high and close together, the model is too simple (high bias). A big gap means overfitting (high variance).

In [ ]:
for approach in ["pretrip", "posttrip"]:
    print(approach)
    display(pd.read_csv(f"results/rf_learning_curve_{approach}.csv").round(3))

## Step 7 (optional): SHAP explanation of the random forest
SHAP shows how each input pushes one trip's predicted fare above or below the average fare. Uses a few hundred test trips to keep it fast.

In [ ]:
RUN_SHAP = True
if RUN_SHAP:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap
    df_feat = lm.add_features(clean)
    _, _, test = lm.chronological_split(df_feat)
    for approach in ["Pre-trip", "Post-trip"]:
        feats = rfm.FEATURE_SETS[approach]
        X_shap = test[feats].sample(300, random_state=rfm.RANDOM_STATE)
        explainer = shap.TreeExplainer(models[("Random forest", approach)])
        sv = explainer(X_shap)
        print(f"Random forest | {approach}")
        shap.plots.beeswarm(sv, max_display=len(feats))

## Limits
- With `SAMPLE_ROWS = 100_000` the trips cover only about a day and a half, so the numbers are not representative. Keep `SAMPLE_ROWS = 0` for results we report.
- To keep the whole month fast, each tree is trained on a bootstrap sample of at most 200,000 training trips (`TREE_MAX_SAMPLES` in `random_forest.py`).
- Zone IDs are used as numbers: trees split them into groups, but nearby ID numbers are not nearby places.
- One month of data says nothing about other seasons, and post-trip inputs (distance, duration) are not available for a quote before the ride.